In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from semevalpolar.finetuning.instruct.reasoning_prompt import run_examples_with_tqdm
from semevalpolar.finetuning.instruct.local_inference import run_local_ollama
from semevalpolar.finetuning.instruct.templates import parse_prompt, run, build_text, parse_prompt_structured
from semevalpolar.llm.data_utils import read_dataset
from semevalpolar.utils import get_project_root
from semevalpolar.llm.main import create_response_from_prompt_file, create_response_ollama


In [ ]:
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv')
df = read_dataset(data_path)

In [ ]:
example = pd.DataFrame({
	"text": df["text"],
	"ground_truth": df["polarization"]
})

example = example.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
response_dict = run_examples_with_tqdm(
	example,
	run_fn=run_local_ollama,
	parse_fn=parse_prompt_structured,
	build_text_fn=build_text,
	prompt_path="prompt-reasoning-v3.txt",
	model="gemma3:27b",
	limit=2
)

In [ ]:
with open("output.jsonl", "w", encoding="utf-8") as f:
    for item in response_dict["dataset"]:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

# Classify Ambiguities

In [ ]:
data_path = Path("data/curation/splits/val.jsonl")
with data_path.open("r", encoding="utf-8") as f:
	records = f.readlines()

In [ ]:
file_path = "response_dict.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

responses = []
for item in tqdm(data["dataset"], desc="Processing responses"):
    text = item["input"]
    reasoning = item["reasoning"]
    label = int(item["final answer (polarization)"])
    response = create_response_ollama(
        "prompt-ambiguous.txt",
        text,
        reasoning,
        label
    )
    responses.append(response)